# From PDF to KPIs, step by step

This notebook follows one annual report (Maroc Telecom RFA 2025) through every stage of the pipeline:

| Step | What exists after it | Who does it |
| --- | --- | --- |
| 1. Source | a PDF: pixels and positioned characters, no structure | AMMC |
| 2. Raw parse | Docling JSON: boxes, text, row and column indices | parser (Docling or Azure) |
| 3. Normalised parse | `ParseResult`: tables of `Cell(row, col, text, kind)`, the same shape for every parser | our adapter |
| 4. Figures (structured output) | `Figure(line_item, period, value_mad, page, row, col)` with a pointer back to the cell | **extraction**: label mapping, scale, period |
| 5. KPIs | ratios computed in Python from Figures, each carrying its inputs | Python, never the LLM |

The golden set (`figures.csv`) is the answer key for step 4. The bake-off (#10) scores steps 2-3; steps 4-5 are shown here as a sketch of what #13 and #7 will build.

Run it from `prototypes/parser-bakeoff/` after `uv run bakeoff fetch` and `uv run bakeoff parse docling`:

```sh
uv run --with jupyter jupyter lab walkthrough.ipynb
```

In [ ]:
import json
import re
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import pypdfium2

from bakeoff.score import is_number, label, load_golden, number, score
from bakeoff.tables import ParseResult, Table

DATA = Path("data")
PDF = DATA / "Maroc_Telecom_RFA_2025.pdf"
GOLDEN = Path("../../golden/maroc-telecom-rfa-2025/figures.csv")
pd.set_option("display.max_colwidth", 60)

## 1. The source: a PDF page

A PDF has no tables. It has characters drawn at coordinates. "36 681" is five glyphs and a space placed somewhere on page 24. Everything that follows is an attempt to recover the structure a human sees at a glance.

In [ ]:
pdf = pypdfium2.PdfDocument(PDF)
print(f"{len(pdf)} pages")
page = pdf[23]  # page 24, zero-based
image = page.render(scale=1.6).to_pil()
width, height = image.size
image.crop((0, int(height * 0.06), width, int(height * 0.62)))

## 2. Raw parser output

Docling turns the page into a JSON document. For a table it gives a list of cells, each with a bounding box, its row and column offsets, spans, and whether it looks like a header. It knows *where* things are, not *what* they mean: "Chiffre d'affaires" is just text in row 1, column 0.

In [ ]:
raw = json.loads((DATA / "raw" / "docling.json").read_text())
raw_table = next(t for t in raw["tables"] if any(p["page_no"] == 24 for p in t["prov"]))
print(f"{len(raw['tables'])} tables in the document, {len(raw_table['data']['table_cells'])} cells in the page-24 table")
revenue_cell = next(c for c in raw_table["data"]["table_cells"] if c["text"] == "36 681")
revenue_cell

## 3. Normalised parse: the same shape for every parser

`parse_docling.py` and `parse_azure.py` each convert their parser's JSON into one `ParseResult`, so everything downstream (scorer, extractor) works on either. A table is just a list of `Cell(row, col, text, kind)`. Put back on a grid, it looks like the page again.

In [ ]:
docling = ParseResult.load(DATA / "parsed" / "docling.json")
print(docling.parser, docling.version, f"{docling.seconds} s", f"{len(docling.tables)} tables")


def grid(table: Table) -> pd.DataFrame:
    """The table's cells laid out by row and column, as the parser saw them."""
    frame = pd.DataFrame([(c.row, c.col, c.text) for c in table.cells], columns=["row", "col", "text"])
    return frame.pivot_table(index="row", columns="col", values="text", aggfunc="first").fillna("")


def tables_on(page: int) -> list[Table]:
    return [t for t in docling.tables if page in t.pages]


income_statement = tables_on(24)[0]
print("caption:", repr(income_statement.caption))
grid(income_statement)

Things to notice, because the extractor has to deal with each of them:

- **The caption is empty.** The scale ("En millions MAD") is inside cell (0, 0), not in a title. Without it, 36 681 could be MAD, thousands or millions.
- **Column 1 is "Note"**: note numbers (17, 18...) that look exactly like values.
- **Labels repeat.** "Résultat net" appears on rows 17 and 21, and "Part du groupe" on rows 22 and 25 with *different meanings*: net income group share (6 968) vs comprehensive income group share (7 083). A label alone is not enough; you need its parent row.
- **Rows 31-32**: "Résultat net dilué par action" got split from its values: a parser defect.

## 4a. Scoring the parse against the golden set

The golden set says, for each figure, where it is and what it should read. The scorer looks for that number in the cited page's tables and checks the row and column labels around it.

In [ ]:
golden = load_golden(GOLDEN)
scores = score(docling, golden)
pd.Series([s.outcome for s in scores]).value_counts()

In [ ]:
misses = pd.DataFrame(
    [
        dict(outcome=s.outcome, page=s.golden.page, line_item=s.golden.line_item, period=s.golden.period,
             value=s.golden.value, row=s.golden.row_label)
        for s in scores
        if s.outcome != "exact"
    ]
)
misses

11 of the 13 misses are on pages 79-81, the company-only accounts in CGNC format. Two rows show why:

- **p. 81**: Docling fused two rows into one. "III- RESULTAT D'EXPLOITATION" and "IV- PRODUITS FINANCIERS" share a label cell, and each value cell holds two numbers ("6 539 241 1 702 429").
- **p. 79**: the cash line's values (341 255, 270 518) were glued onto the "TOTAL II" row above it.

No extractor can recover a value the parse has already put in the wrong cell. That is why the parser is chosen on a scored bake-off.

In [ ]:
def rows_containing(page: int, text: str) -> pd.DataFrame:
    return pd.concat(
        grid(t)[lambda g: g.apply(lambda row: row.str.contains(text, regex=False).any(), axis=1)]
        for t in tables_on(page)
    )


pd.concat({"p. 81": rows_containing(81, "RESULTAT D'EXPLOITATION"), "p. 79": rows_containing(79, "TOTAL II (F+G+H+I)")})

## 4b. Extraction: from cells to Figures

This is the step you asked about: **how a grid of strings becomes structured data**. For each cell, the extractor answers three questions:

1. **What is it?** Map the row label to a line item from our fixed vocabulary (`revenue`, `ebitda`, ...). Here that's a hand-written dictionary; in the product this is the LLM's job, constrained to the vocabulary, because real labels vary ("Chiffre d'affaires", "CA", "Produits des activités ordinaires"...).
2. **For when?** Parse the column header into a period ("31-déc.-25" and "12M 2025" both mean FY2025; "T4 2025" and "Variation" are skipped).
3. **In what unit?** Find the table's scale ("millions", "milliers") and convert to MAD.

The number itself is **never generated**: it is copied from the cell text, and the Figure keeps the page, row and column, so anyone can check it against the source.

In [ ]:
LABELS = {  # normalised row label -> line item; the LLM's job in the product
    "chiffre d affaires": "revenue",
    "resultat operationnel": "operating_income_ebit",
    "resultat net": "net_income",
    "resultat net > part du groupe": "net_income_group_share",
    "resultat net global > part du groupe": "comprehensive_income_group_share",
    "ebitda": "ebitda",
    "capex": "capex",
    "dont frequences et licences": "capex_licences",
    "dette nette": "net_debt",
    "flux nets de tresorerie provenant des activites d exploitation": "operating_cash_flow",
}
CHILD_LABELS = {"part du groupe", "interets minoritaires"}  # meaningless without their parent row
SCALES = {"millions": 1_000_000, "millier": 1_000, "milliers": 1_000}


@dataclass
class Figure:
    line_item: str
    period: str
    value_mad: int
    raw_label: str
    cell_text: str
    page: int
    row: int
    col: int


def period(header: str) -> str | None:
    if match := re.fullmatch(r"31-déc\.?-(\d{2})", header.strip()):
        return f"FY20{match[1]}"
    if match := re.fullmatch(r"12M (\d{4})", header.strip()):
        return f"FY{match[1]}"
    return None  # "Note", "T4 2025", "Variation"...


def scale(table: Table) -> int:
    text = " ".join([table.caption] + [c.text for c in table.cells if c.row == 0]).lower()
    return next((factor for word, factor in SCALES.items() if word in text), 1)


def extract(table: Table) -> list[Figure]:
    cells = {(c.row, c.col): c.text for c in table.cells}
    header_row = min(c.row for c in table.cells)
    periods = {col: period(text) for (row, col), text in cells.items() if row == header_row}
    factor = scale(table)
    figures, parent = [], ""
    for row in sorted({c.row for c in table.cells} - {header_row}):
        raw_label = cells.get((row, 0), "")
        key = label(raw_label)
        if key in CHILD_LABELS:
            key = f"{parent} > {key}"
        elif key:
            parent = key
        line_item = LABELS.get(key)
        if not line_item:
            continue
        for col, when in periods.items():
            text = cells.get((row, col), "")
            if when and is_number(text) and "%" not in text:
                value = round(float(number(text).replace(",", ".")) * factor)
                figures.append(Figure(line_item, when, value, raw_label, text, table.pages[0], row, col))
    return figures


figures = [f for page in (9, 24, 25) for table in tables_on(page) for f in extract(table)]
extracted = pd.DataFrame(figures).drop_duplicates(["line_item", "period", "value_mad", "page"])
extracted

Notice revenue **FY2024 comes out twice with different values**: 36 699 from the income statement (p. 24) and 36 704 from the adjusted results table (p. 9). Both are correct readings of different passages on different bases. That is why the ontology (#8) stores one Fact per Passage, with its source, and does not overwrite one with the other.

### Checking the extraction against the golden set

Now the answer key earns its keep: join on line item, period and page, and compare values.

In [ ]:
answer_key = pd.read_csv(GOLDEN)[["line_item", "period", "page", "value_mad"]]
check = extracted.merge(answer_key, on=["line_item", "period", "page"], how="left", suffixes=("", "_golden"))
check["value_mad_golden"] = check.value_mad_golden.astype("Int64")  # whole MAD amounts, or missing
check["check"] = [
    "not in golden v0" if pd.isna(expected) else "match" if value == expected else "DIFFERS"
    for value, expected in zip(check.value_mad, check.value_mad_golden)
]
print(check.check.value_counts().to_string())
check[["line_item", "period", "page", "cell_text", "value_mad", "value_mad_golden", "check"]]

"not in golden v0" means the answer key doesn't list that figure on that page: the extraction isn't wrong, it just can't be checked yet. Every figure that *is* in the key matches.

## 5. KPIs: Python computes, from Figures only

A KPI is a formula over Figures. It is computed in Python, never by the LLM, and it keeps the Figures it used, so every ratio traces back to cells on pages. Per the #7 rules, a KPI is only shown once its inputs are checked by a Review; nothing here has been reviewed, so all of them would stay hidden in the app.

In [ ]:
def pick(line_item: str, page: int, when: str = "FY2025") -> Figure:
    return next(f for f in figures if f.line_item == line_item and f.page == page and f.period == when)


revenue = pick("revenue", 24)          # income statement
net_income = pick("net_income", 24)
cash_flow = pick("operating_cash_flow", 25)
ebitda = pick("ebitda", 9)             # adjusted results table; EBITDA isn't an IFRS line
capex = pick("capex", 9)
licences = pick("capex_licences", 9)
net_debt = pick("net_debt", 9)

KPIS = [
    ("EBITDA margin", lambda: ebitda.value_mad / revenue.value_mad, [ebitda, revenue], "50,4%"),
    ("Net margin", lambda: net_income.value_mad / revenue.value_mad, [net_income, revenue], None),
    ("Cash conversion (CFO / net income)", lambda: cash_flow.value_mad / net_income.value_mad, [cash_flow, net_income], None),
    ("Capex excl. licences / revenue", lambda: (capex.value_mad - licences.value_mad) / revenue.value_mad,
     [capex, licences, revenue], "25,6%"),
    ("Net debt / EBITDA", lambda: net_debt.value_mad / ebitda.value_mad, [net_debt, ebitda], "0,9x"),
]

pd.DataFrame(
    [
        dict(kpi=name, value=round(formula(), 4), reported_by_company=reported, status="unchecked",
             inputs="; ".join(f"{f.line_item} p{f.page} r{f.row}c{f.col} '{f.cell_text}'" for f in inputs))
        for name, formula, inputs, reported in KPIS
    ]
)

Where the company prints its own ratio on page 9, we can cross-check:

- **EBITDA margin** 0.5042 matches the printed 50,4%.
- **Capex excl. licences / revenue** 0.2558 matches 25,6%, but only because we subtracted licences. Plain capex / revenue would be 29.7%: definitions matter, which is why #7 versions every KPI rule.
- **Net debt / EBITDA** 0.952 against a printed 0,9x. Rounding would give 1,0x, so the company either truncates or uses a different EBITDA or net debt. This is exactly the kind of discrepancy a Review should catch before the KPI is shown.

## Summary

```
PDF ──parser──▶ raw JSON ──adapter──▶ ParseResult ──extract──▶ Figures ──Python──▶ KPIs
                (boxes)               (grid of text)  (label, period,  (cited cells)  (formula + inputs)
                                                       scale)
       scored by the bake-off (#10)    checked against figures.csv     checked by Reviews (#7)
```

- **Parsing** recovers the grid. If it merges cells or shifts columns (pp. 79-81), nothing later can fix it, hence the bake-off.
- **Extraction** gives cells meaning: line item, period, unit. The label mapping is the only fuzzy part and the place for the LLM; the number is always copied, and the Figure points back to its cell.
- **KPIs** are deterministic Python over Figures, versioned, and shown only once their inputs are reviewed.